In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

from src.data import (
    load_sales, load_web_traffic, 
    load_promotions, load_order_items, 
    load_orders
)
from src.features import add_time_features, add_lag_features, list_feature_columns
from src.pipelines.infer import fit_and_predict
from src.models import SklearnRegressorConfig, SklearnRegressorWrapper
from src.utils import prepare_submission
import pandas as pd


DATA_ROOT = project_root / "data/datathon-2026-round-1"

Project root: D:\MyML\datathon2026
Source path added: D:\MyML\datathon2026\src


In [2]:
TRAIN_RANGE = ("2013-01-01", "2022-12-31")
PREDICT_RANGE = ("2023-01-01", "2024-07-01")
df_sales = load_sales(data_root=DATA_ROOT)
df_traffic = load_web_traffic(data_root=DATA_ROOT)

df_traffic_test = pd.read_csv(f"{project_root}/results/webtraffic_predictions.csv", parse_dates=["date"])
df_traffic_test.set_index("date", inplace=True)

In [3]:
TRAFFIC_FEATURES_RAW = ["date", "sessions", "unique_visitors"]
SALE_FEATURES_RAW = ["date", "Revenue", "COGS"]

In [4]:
df = pd.DataFrame(
    {"date": pd.date_range(start=TRAIN_RANGE[0], end=PREDICT_RANGE[1], freq="D")}
)
df = df.merge(df_sales[SALE_FEATURES_RAW], on="date", how="left")
df = df.merge(df_traffic[TRAFFIC_FEATURES_RAW], on="date", how="left")
df.set_index("date", inplace=True)
df.update(df_traffic_test[TRAFFIC_FEATURES_RAW[1:]])
df.reset_index(inplace=True)

In [5]:
df = add_time_features(df, date_col="date")

In [6]:
model_config = SklearnRegressorConfig(
    model_type="lightgbm",
)

submission = fit_and_predict(
    df=df,
    model_config=model_config,
    train_range=TRAIN_RANGE,
    predict_range=PREDICT_RANGE,
)
submission.head()

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001982 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 888
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 11
[LightGBM] [Info] Start training from score 4295996.395203
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001014 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 888
[LightGBM] [Info] Number of data points in the train set: 3652, number of used features: 11
[LightGBM] [Info] Start training from score 1.157164


,Date,Revenue,COGS
0,2023-01-01,2.518074e+06,2.532460e+06
1,2023-01-02,1.817012e+06,1.632927e+06
2,2023-01-03,1.163964e+06,1.001203e+06
3,2023-01-04,9.645782e+05,8.204835e+05
4,2023-01-05,9.697595e+05,8.172956e+05


In [9]:
submission.to_csv(project_root / "submissions/lightgbm_1.csv", index=False)